# 📝 Advanced Python (Decorators, Generators, Iterators, Context Managers, Dataclasses)
### Exercises & Solutions — 28 Problems

This notebook is exercises-and-solutions only. It assumes you've already covered the
concept notebook for this topic. Each problem targets a **distinct function, pattern,
or real-world scenario** so that working through all of them gives you practical
exposure to everything commonly used on the job.

**Coverage map:**

- Iterators & the iterator protocol (1-4)
- Generators: yield, yield from, send, generator expressions (5-10)
- Decorators: simple, parameterized, stacking, class-based, functools (11-18)
- Context managers: class-based and @contextmanager (19-23)
- Dataclasses: fields, defaults, frozen, ordering, post_init (24-28)


---


### 1. Manual Iterator Protocol

Build a `Range2` class replicating `range()` behaviour using the manual iterator protocol (`__iter__` + `__next__`).

In [ ]:
class Range2:
    def __init__(self, start, stop, step=1):
        self.start, self.stop, self.step = start, stop, step
    def __iter__(self):
        self.current = self.start
        return self
    def __next__(self):
        if (self.step > 0 and self.current >= self.stop) or (self.step < 0 and self.current <= self.stop):
            raise StopIteration
        val = self.current
        self.current += self.step
        return val

print(list(Range2(0, 10, 2)))
print(list(Range2(10, 0, -3)))

### 2. iter() with a Sentinel Value

Use the two-argument form of `iter(callable, sentinel)` to read from a simulated data source until a sentinel value appears.

In [ ]:
data_source = iter([1, 2, 3, 0, 4, 5])   # 0 acts as our "end" signal once reached

def read_next():
    return next(data_source)

# iter(callable, sentinel) keeps calling callable() until it returns sentinel
results = list(iter(read_next, 0))
print(results)   # stops BEFORE the 0, doesn't include it

### 3. Custom Iterable vs Iterator Distinction

Build a `Deck` class that is ITERABLE (has `__iter__`) but returns a NEW iterator object each time, proving it can be iterated multiple times independently (unlike an exhausted generator).

In [ ]:
class DeckIterator:
    def __init__(self, cards):
        self.cards = cards
        self.index = 0
    def __next__(self):
        if self.index >= len(self.cards):
            raise StopIteration
        card = self.cards[self.index]
        self.index += 1
        return card
    def __iter__(self):
        return self

class Deck:
    def __init__(self, cards):
        self.cards = cards
    def __iter__(self):
        return DeckIterator(self.cards)    # fresh iterator each call!

deck = Deck(["A♠", "K♥", "Q♦"])
print(list(deck))
print(list(deck))   # works again! NOT exhausted, unlike a plain generator object

### 4. itertools.islice for Lazy Slicing

Use `itertools.islice` to get items 3 through 7 of an infinite generator WITHOUT materializing the whole sequence.

In [ ]:
from itertools import islice, count

infinite = count(start=1, step=1)         # 1, 2, 3, 4, 5, ... forever
sliced = list(islice(infinite, 3, 8))     # indices 3 to 7 (5 items)
print(sliced)

# Real use case: paginate an infinite/huge stream without loading it all
def event_stream():
    i = 0
    while True:
        yield f"event-{i}"
        i += 1

page_2 = list(islice(event_stream(), 10, 15))   # "page 2" of 5 items
print(page_2)

### 5. Basic Generator Function

Write a generator `evens_up_to(n)` yielding even numbers from 0 to n, and show it uses O(1) memory regardless of n by comparing `sys.getsizeof` of the generator vs an equivalent list.

In [ ]:
import sys

def evens_up_to(n):
    for i in range(0, n+1, 2):
        yield i

gen = evens_up_to(1_000_000)
lst = list(range(0, 1_000_001, 2))
print(f"Generator size: {sys.getsizeof(gen)} bytes")
print(f"List size: {sys.getsizeof(lst):,} bytes")
print(f"First 5 from generator: {[next(gen) for _ in range(5)]}")

### 6. Generator Expression vs List Comprehension Memory

Compare building a generator expression vs a list comprehension for filtering a large range, showing the generator is lazy (doesn't run until iterated).

In [ ]:
def expensive_check(x):
    print(f"  checking {x}")   # side effect to PROVE when evaluation happens
    return x % 2 == 0

print("Creating generator expression (should print NOTHING yet):")
gen_exp = (x for x in range(5) if expensive_check(x))
print("Generator created, no checks ran yet!\n")

print("Now consuming it:")
print(list(gen_exp))

### 7. yield from for Generator Delegation

Write `flatten(nested_list)` using `yield from` recursively to flatten arbitrarily nested lists into a flat generator.

In [ ]:
def flatten(items):
    for item in items:
        if isinstance(item, list):
            yield from flatten(item)    # delegates to a sub-generator
        else:
            yield item

nested = [1, [2, 3, [4, 5, [6]], 7], 8, [9, [10]]]
print(list(flatten(nested)))

### 8. Generator with send() for Coroutine-style Communication

Build a generator-based accumulator that receives values via `.send()` and yields the running total, demonstrating two-way generator communication.

In [ ]:
def accumulator():
    total = 0
    while True:
        value = yield total
        total += value

acc = accumulator()
next(acc)                        # prime the generator (advances to first yield)
print(acc.send(10))               # 10
print(acc.send(5))                # 15
print(acc.send(-3))               # 12

### 9. Generator .throw() and .close()

Demonstrate `.throw()` to inject an exception into a running generator, and `.close()` to terminate it cleanly, observing `GeneratorExit` handling.

In [ ]:
def resource_generator():
    try:
        while True:
            yield "resource acquired"
    except GeneratorExit:
        print("  Cleanup: releasing resource (GeneratorExit caught)")
    except ValueError as e:
        print(f"  Handled injected error: {e}")
        yield "recovered"

gen = resource_generator()
print(next(gen))
print(gen.throw(ValueError, "simulated failure"))   # injects exception INTO the generator
gen.close()                                          # triggers GeneratorExit cleanup
print("Generator closed cleanly")

### 10. Building a Lazy Pipeline with Chained Generators

Build a 3-stage lazy pipeline (`read_lines` → `parse_numbers` → `filter_positive`) where NOTHING executes until the final result is consumed, processing one item at a time end-to-end.

In [ ]:
def read_lines(data):
    for line in data:
        yield line

def parse_numbers(lines):
    for line in lines:
        try:
            yield int(line)
        except ValueError:
            continue   # skip malformed lines

def filter_positive(numbers):
    for n in numbers:
        if n > 0:
            yield n

raw_data = ["5", "-3", "oops", "10", "-1", "7"]
pipeline = filter_positive(parse_numbers(read_lines(raw_data)))
print(list(pipeline))   # only NOW does the whole chain actually execute, lazily, item by item

### 11. Basic Logging Decorator

Write a `@log_calls` decorator printing the function name, args, and return value every time it's called.

In [ ]:
import functools

def log_calls(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        print(f"{func.__name__}(args={args}, kwargs={kwargs}) -> {result}")
        return result
    return wrapper

@log_calls
def add(a, b):
    return a + b

add(3, 4)
add(a=10, b=20)

### 12. Timing Decorator with Statistics

Write a `@timed` decorator that records EVERY call's duration on the function object itself (`func.call_times`), allowing later inspection of performance stats.

In [ ]:
import functools, time

def timed(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        wrapper.call_times.append(elapsed)
        return result
    wrapper.call_times = []
    return wrapper

@timed
def work(n):
    return sum(range(n))

for n in [1000, 10000, 100000]:
    work(n)
print(f"Recorded {len(work.call_times)} calls")
print(f"Avg time: {sum(work.call_times)/len(work.call_times)*1000:.4f}ms")

### 13. Parameterized Decorator (Decorator Factory)

Write `@validate_types(a=int, b=int)` — a decorator FACTORY that validates argument types based on a per-decoration configuration.

In [ ]:
import functools, inspect

def validate_types(**expected_types):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            bound = inspect.signature(func).bind(*args, **kwargs)
            bound.apply_defaults()
            for name, expected in expected_types.items():
                value = bound.arguments.get(name)
                if value is not None and not isinstance(value, expected):
                    raise TypeError(f"{name} must be {expected.__name__}, got {type(value).__name__}")
            return func(*args, **kwargs)
        return wrapper
    return decorator

@validate_types(a=int, b=int)
def divide(a, b):
    return a / b

print(divide(10, 2))
try:
    divide(10, "2")
except TypeError as e:
    print("Blocked:", e)

### 14. Stacking Multiple Decorators — Order Matters

Stack 3 decorators (`@bold`, `@italic`, `@underline`) on a function returning plain text, and show how the OUTPUT changes depending on stacking order.

In [ ]:
def bold(func):
    def wrapper(*a, **kw): return f"**{func(*a, **kw)}**"
    return wrapper
def italic(func):
    def wrapper(*a, **kw): return f"_{func(*a, **kw)}_"
    return wrapper
def underline(func):
    def wrapper(*a, **kw): return f"__{func(*a, **kw)}__"
    return wrapper

@bold
@italic
@underline
def get_text():
    return "Hello"

print("bold(italic(underline(x))):", get_text())

@underline
@italic
@bold
def get_text_v2():
    return "Hello"

print("underline(italic(bold(x))):", get_text_v2())
print("Decorators apply bottom-up, closest to the function runs FIRST")

### 15. Caching Decorator (Manual, then functools.lru_cache)

Implement a manual memoization decorator, then show `functools.lru_cache` does the same thing with built-in cache size limiting and `.cache_info()`.

In [ ]:
import functools, time

def manual_cache(func):
    store = {}
    @functools.wraps(func)
    def wrapper(*args):
        if args not in store:
            store[args] = func(*args)
        return store[args]
    return wrapper

@manual_cache
def slow_square(n):
    time.sleep(0.01)
    return n * n

start = time.perf_counter()
slow_square(5); slow_square(5); slow_square(5)
print(f"Manual cache (3 calls, 1 unique): {time.perf_counter()-start:.4f}s")

@functools.lru_cache(maxsize=128)
def fib(n):
    return n if n < 2 else fib(n-1) + fib(n-2)

fib(30)
print("lru_cache info:", fib.cache_info())

### 16. Retry Decorator with Exponential Backoff

Write `@retry_with_backoff(max_attempts=4, base_delay=0.01)` that retries a failing function with delay doubling each attempt.

In [ ]:
import functools, time

def retry_with_backoff(max_attempts=3, base_delay=0.1):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            delay = base_delay
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    print(f"  Attempt {attempt} failed: {e} (next delay: {delay:.3f}s)")
                    if attempt == max_attempts:
                        raise
                    time.sleep(delay)
                    delay *= 2          # exponential backoff
        return wrapper
    return decorator

state = {"n": 0}
@retry_with_backoff(max_attempts=4, base_delay=0.01)
def flaky():
    state["n"] += 1
    if state["n"] < 4:
        raise ConnectionError("network blip")
    return "success"

print(flaky())

### 17. Class-Based Decorator with __call__ and State

Build a `RateLimiter` class-based decorator limiting a function to N calls, raising after the limit, demonstrating decorators implemented as classes (not just functions).

In [ ]:
import functools

class RateLimiter:
    def __init__(self, max_calls):
        self.max_calls = max_calls
    def __call__(self, func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            wrapper.calls += 1
            if wrapper.calls > self.max_calls:
                raise RuntimeError(f"{func.__name__} exceeded {self.max_calls} calls")
            return func(*args, **kwargs)
        wrapper.calls = 0
        return wrapper

@RateLimiter(max_calls=3)
def api_call():
    return "ok"

for i in range(3):
    print(api_call())
try:
    api_call()
except RuntimeError as e:
    print("Blocked:", e)

### 18. functools.singledispatch for Type-Based Overloading

Use `@functools.singledispatch` to implement a `describe()` function with different behaviour per ARGUMENT TYPE (simulating function overloading, which Python doesn't have natively).

In [ ]:
from functools import singledispatch

@singledispatch
def describe(value):
    return f"Unknown type: {type(value).__name__}"

@describe.register
def _(value: int):
    return f"Integer: {value} (even={value % 2 == 0})"

@describe.register
def _(value: str):
    return f"String: '{value}' (length={len(value)})"

@describe.register
def _(value: list):
    return f"List with {len(value)} items"

for v in [42, "hello", [1,2,3], 3.14]:
    print(describe(v))

### 19. Class-Based Context Manager with State Tracking

Build a `Stopwatch` class-based context manager that tracks total accumulated time across MULTIPLE `with` blocks using the same instance.

In [ ]:
import time

class Stopwatch:
    def __init__(self):
        self.total = 0
    def __enter__(self):
        self.start = time.perf_counter()
        return self
    def __exit__(self, *exc_info):
        self.total += time.perf_counter() - self.start
        return False

sw = Stopwatch()
with sw:
    time.sleep(0.02)
with sw:
    time.sleep(0.03)
print(f"Total accumulated time across both blocks: {sw.total:.3f}s")

### 20. Context Manager Suppressing Specific Exceptions

Build a context manager `ignore(*exception_types)` (similar to `contextlib.suppress`) implemented from scratch using `__exit__`'s return value.

In [ ]:
class ignore:
    def __init__(self, *exception_types):
        self.exception_types = exception_types
    def __enter__(self):
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is not None and issubclass(exc_type, self.exception_types):
            print(f"  Ignored {exc_type.__name__}: {exc_val}")
            return True   # suppress
        return False

with ignore(ZeroDivisionError, KeyError):
    1 / 0
print("Continued after suppressed ZeroDivisionError")

# Prove it's TYPE-SPECIFIC: only listed types are suppressed, others still propagate
try:
    with ignore(ZeroDivisionError):    # KeyError is NOT in the ignore list
        {}["missing"]
except KeyError as e:
    print(f"As expected, KeyError was NOT suppressed and propagated: {e!r}")


### 21. @contextmanager Decorator for Function-Based CMs

Use `contextlib.contextmanager` to build a `temporary_attribute(obj, name, value)` context manager that temporarily overrides an object's attribute and restores it afterward.

In [ ]:
from contextlib import contextmanager

@contextmanager
def temporary_attribute(obj, name, value):
    had_attr = hasattr(obj, name)
    old_value = getattr(obj, name, None)
    setattr(obj, name, value)
    try:
        yield obj
    finally:
        if had_attr:
            setattr(obj, name, old_value)
        else:
            delattr(obj, name)

class Config:
    debug = False

print("Before:", Config.debug)
with temporary_attribute(Config, "debug", True):
    print("Inside:", Config.debug)
print("After (restored):", Config.debug)

### 22. Nested and Multiple Context Managers

Combine MULTIPLE context managers in one `with` statement (comma syntax) and show the nesting/exit order using a simple logging context manager.

In [ ]:
from contextlib import contextmanager

@contextmanager
def step(name):
    print(f"ENTER {name}")
    yield name
    print(f"EXIT  {name}")

with step("outer"), step("inner"):
    print("  doing work in the middle")
# Exit order should be reverse of entry: inner exits BEFORE outer

### 23. ExitStack for Dynamic Number of Context Managers

Use `contextlib.ExitStack` to manage a DYNAMIC (runtime-determined) number of context managers — e.g. opening N temp resources, all cleaned up correctly even if one fails midway.

In [ ]:
from contextlib import contextmanager, ExitStack

@contextmanager
def resource(name):
    print(f"Acquiring {name}")
    yield name
    print(f"Releasing {name}")

resource_names = ["db", "cache", "logger"]

with ExitStack() as stack:
    resources = [stack.enter_context(resource(name)) for name in resource_names]
    print(f"All acquired: {resources}")
# ExitStack closes them all in REVERSE order automatically, even though
# the number of resources was only known at runtime

### 24. Basic Dataclass with Defaults and default_factory

Build a `Task` dataclass with a default `priority=3` and a `tags: List[str]` using `field(default_factory=list)` to avoid the mutable-default trap.

In [ ]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class Task:
    title: str
    priority: int = 3
    tags: List[str] = field(default_factory=list)

t1 = Task("Fix bug")
t2 = Task("Write docs", priority=1)
t1.tags.append("urgent")
print(t1, t2)
print("t2.tags independent of t1.tags:", t2.tags)   # [] - NOT shared, thanks to default_factory

### 25. Frozen (Immutable) Dataclass + Hashability

Build a `frozen=True` dataclass `Coordinate` and show it becomes hashable automatically (usable in sets/dict keys), unlike a normal mutable dataclass.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Coordinate:
    lat: float
    lon: float

c1 = Coordinate(40.7, -74.0)
c2 = Coordinate(40.7, -74.0)
print(c1 == c2)                     # value equality, auto-generated
print(len({c1, c2}))                 # 1 - frozen dataclasses are hashable, dedups correctly

try:
    c1.lat = 0
except Exception as e:
    print("Blocked mutation:", type(e).__name__)

### 26. order=True for Auto-Generated Comparisons

Build an `order=True` dataclass `Version(major, minor, patch)` and show it sorts naturally by FIELD ORDER without writing `__lt__` manually.

In [ ]:
from dataclasses import dataclass

@dataclass(order=True)
class Version:
    major: int
    minor: int
    patch: int
    def __str__(self):
        return f"{self.major}.{self.minor}.{self.patch}"

versions = [Version(1,2,0), Version(2,0,0), Version(1,0,5), Version(1,2,3)]
for v in sorted(versions):
    print(v)
print("1.0.0 < 1.0.1:", Version(1,0,0) < Version(1,0,1))

### 27. __post_init__ for Derived Fields & Validation

Build a `Rectangle` dataclass that validates `width`/`height` > 0 in `__post_init__` AND computes a derived `area` field afterward (using `field(init=False)`).

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Rectangle:
    width: float
    height: float
    area: float = field(init=False)     # not part of __init__ signature

    def __post_init__(self):
        if self.width <= 0 or self.height <= 0:
            raise ValueError("width and height must be positive")
        self.area = self.width * self.height

r = Rectangle(4, 5)
print(r)
try:
    Rectangle(-1, 5)
except ValueError as e:
    print("Blocked:", e)

### 28. Dataclass Inheritance & asdict()/astuple() conversion

Build a base `Person` dataclass and an `Employee(Person)` subclass adding fields, then convert instances to dict/tuple via `dataclasses.asdict`/`astuple`.

In [ ]:
from dataclasses import dataclass, asdict, astuple

@dataclass
class Person:
    name: str
    age: int

@dataclass
class Employee(Person):           # inherits name, age; adds salary
    salary: float = 0.0

e = Employee("Alice", 30, 95000)
print(e)
print("asdict:", asdict(e))
print("astuple:", astuple(e))
print("isinstance Person:", isinstance(e, Person))   # True - real inheritance